# Autoregressive 模型评测

加载 canonical artifact，比较随机生成质量、surprisal 和搜索覆盖率。

## 加载模型和测试集

In [1]:
from dataclasses import asdict
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from scripts.pipeline import get_device
from scripts.evaluation import (
    coverage_curve,
    evaluate_random_generation_with_coverage,
    plot_coverage_curve,
    plot_efficiency_curve,
    plot_generation_quality,
    plot_random_coverage_curve,
    plot_random_efficiency_curve,
    plot_surprisal_boxplot,
    plot_surprisal_histogram,
    read_test_passwords,
    summarize_surprisal,
    save_coverage_npz,
    save_surprisal_npz
)
from scripts.inference import (
    best_first_search,
    load_generation_candidates,
    load_inference_model,
    score_passwords,
)
device = get_device()
data_dir = Path("data/processed")
tokenizer_dir = Path("data/processed/tokenizer.json")
gru_dir = Path("output/low/gru")
tcn_dir = Path("output/low/tcn")
transformer_dir = Path("output/low/transformer")
bigram_dir = Path("output/bigram")
output_dir = Path("output/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)
test_passwords = read_test_passwords(data_dir / "test.txt", seed=42)
gru, tokenizer = load_inference_model(gru_dir / "model.pt",
                                      tokenizer_dir,
                                      gru_dir / "inference.json",
                                      device=device)
tcn, _ = load_inference_model(tcn_dir / "model.pt",
                              tokenizer_dir,
                              tcn_dir / "inference.json",
                              device=device)
transformer, _ = load_inference_model(transformer_dir / "model.pt",
                                      tokenizer_dir,
                                      transformer_dir / "inference.json",
                                      device=device)
bigram, _ = load_inference_model(bigram_dir / "model.pt",
                                 tokenizer_dir,
                                 bigram_dir / "inference.json",
                                 device=device)
models = {"Bigram": bigram,
          "GRU": gru,
          "TCN": tcn,
          "Transformer": transformer}

## Surprisal 分布

In [ ]:
surprisal = {name: score_passwords(model, tokenizer, test_passwords,
                                   batch_size=10_000, verbose=True)
             for name, model in models.items()}
surprisal_per_token = {name: [sp / (len(pw)+1)
                              for sp, pw in zip(values, test_passwords)]
                       for name, values in surprisal.items()}
summaries = {name: asdict(summarize_surprisal(test_passwords, values))
             for name, values in surprisal.items()}
(output_dir / "surprisal_summary.json").write_text(json.dumps(summaries, indent=2), encoding="utf-8")
for name in models:
    save_surprisal_npz(
        output_dir / f"{name.lower()}_surprisal.npz",
        surprisal[name],
        surprisal_per_token[name],
        max_points=100_000,
    )

score batch 1/388 completed=10000/3872985
score batch 2/388 completed=20000/3872985
score batch 3/388 completed=30000/3872985
score batch 4/388 completed=40000/3872985
score batch 5/388 completed=50000/3872985
score batch 6/388 completed=60000/3872985
score batch 7/388 completed=70000/3872985
score batch 8/388 completed=80000/3872985
score batch 9/388 completed=90000/3872985
score batch 10/388 completed=100000/3872985
score batch 11/388 completed=110000/3872985
score batch 12/388 completed=120000/3872985
score batch 13/388 completed=130000/3872985
score batch 14/388 completed=140000/3872985
score batch 15/388 completed=150000/3872985
score batch 16/388 completed=160000/3872985
score batch 17/388 completed=170000/3872985
score batch 18/388 completed=180000/3872985
score batch 19/388 completed=190000/3872985
score batch 20/388 completed=200000/3872985
score batch 21/388 completed=210000/3872985
score batch 22/388 completed=220000/3872985
score batch 23/388 completed=230000/3872985
score 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_surprisal_histogram(surprisal, ax=axes[0, 0])
plot_surprisal_boxplot(surprisal, ax=axes[0, 1])
plot_surprisal_histogram(surprisal_per_token, ax=axes[1, 0], value_label="Surprisal (bits/token)")
plot_surprisal_boxplot(surprisal_per_token, ax=axes[1, 1], value_label="Surprisal (bits/token)")
fig.tight_layout()
fig.savefig(output_dir / "surprisal.png")

## 随机生成覆盖率

In [ ]:
random_results = {}
random_quality = {}
for name, model in models.items():
    evaluation = evaluate_random_generation_with_coverage(
        model, test_passwords, num_samples=1000_0000, batch_size=10_0000,
        max_length=12, generator=torch.Generator(model.device).manual_seed(2026),
        checkpoint_step=100, verbose=True
    )
    random_results[name] = evaluation.coverage
    random_quality[name] = evaluation.quality
    save_coverage_npz(
        output_dir / f"{name.lower()}_random_coverage.npz",
        evaluation.coverage,
        test_size=len(test_passwords),
    )

In [ ]:
generation_quality = {name: asdict(quality) for name, quality in random_quality.items()}
(output_dir / "generation_quality.json").write_text(
    json.dumps(generation_quality, indent=2), encoding="utf-8"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_generation_quality(random_quality, ax=axes[0])
plot_random_coverage_curve(random_results, ax=axes[1])
plot_random_efficiency_curve(random_results, ax=axes[2])
fig.tight_layout()
fig.savefig(output_dir / "random_generation.png")

## Best-first 搜索覆盖率

In [ ]:
load_gened = False
search_results = {}
for name, model in models.items():
    path = output_dir / f"{name.lower()}_best_first.json"
    candidates = (
        load_generation_candidates(path) if path.exists() and load_gened else best_first_search(
            model, tokenizer, num_candidates=1000_0000, max_length=12,
            node_top_k=5, depth_beam_width=100_0000, save_path=path, verbose=True,
        )
    )
    points = coverage_curve(candidates, test_passwords, checkpoint_step=100)
    search_results[name] = points
    save_coverage_npz(
        output_dir / f"{name.lower()}_best_first_coverage.npz",
        points,
        test_size=len(test_passwords),
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_coverage_curve(search_results, ax=axes[0])
plot_efficiency_curve(search_results, ax=axes[1])
fig.tight_layout()
fig.savefig(output_dir / "search_coverage.png")